<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Image embeddings

**[Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/)** · Vision AI: See, Count and Inspect · Module 3, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn each photo into an embedding with a pretrained model, find the most similar photos, draw the catalogue on a map, and find the classes that the model confuses.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells and the model (47 MB). No account and no key. |
| **You should know** | Transfer learning, from [Transfer learning](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M03-L01-image-embeddings/image-embeddings-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A pretrained model as a feature maker

ResNet-18 learned from more than a million ImageNet photos. We remove its last layer: what is left turns a photo into a list of 512 numbers, an **embedding**. Photos that look alike get close lists. The next cell downloads the model (47 MB) once and embeds the training and test photos. It takes about 20 seconds.

In [ ]:
import torch, timm
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision so that everyone gets the same numbers
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
model = timm.create_model("resnet18.a1_in1k", pretrained=True, num_classes=0, pretrained_cfg_overlay={"file": weights}).eval()
config = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**config)

def embed(paths, batch=32):
    """One list of 512 numbers for each photo, scaled to length 1."""
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[i:i + batch]])
            out.append(model(x).numpy())
    X = np.vstack(out)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
Xtr = embed(list("data/" + train["file"]))
Xte = embed(list("data/" + test["file"]))
print(Xtr.shape, Xte.shape)

> **Check.** You should see `(453, 512) (160, 512)`.

## 2. The most similar photos

Two embeddings of length 1 have a **cosine similarity** from -1 to 1: their dot product. The next cell shows one test photo and the five training photos with the highest similarity.

> **Predict.** Will all five neighbours have the same label as the test photo?

In [ ]:
i = 5
sims = Xtr @ Xte[i]
top = np.argsort(-sims)[:5]
show(["data/" + test["file"][i]] + list("data/" + train["file"][top]),
     [f"test: {test['label'][i]}"] + [f"{train['label'][j]} {sims[j]:.2f}" for j in top])

## 3. A map of the catalogue

512 numbers are too many to draw. PCA keeps the two directions in which the photos differ most. The course also gives you the embeddings of a larger model, CLIP ViT-B/32, in `data/embeddings/clip.npz`: compare the two maps.

In [ ]:
from sklearn.decomposition import PCA
clip = np.load("data/embeddings/clip.npz")
Xclip = np.stack([clip[i] for i in train["image_id"]]).astype(float)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, X, title in ((axes[0], Xtr, "ResNet-18"), (axes[1], Xclip, "CLIP ViT-B/32")):
    xy = PCA(2, random_state=0).fit_transform(X)
    for label in CLASSES:
        m = train["label"] == label
        ax.scatter(xy[m, 0], xy[m, 1], s=8, label=label)
    ax.set_title(title)
axes[1].legend(fontsize=7); plt.show()

## 4. Which classes does the model confuse?

The centre of a class is the mean of its embeddings. Two classes with close centres are hard to tell apart.

In [ ]:
def centres(X, labels):
    C = np.stack([X[np.array(labels) == k].mean(0) for k in CLASSES])
    return C / np.linalg.norm(C, axis=1, keepdims=True)

Cclip = centres(Xclip / np.linalg.norm(Xclip, axis=1, keepdims=True), train["label"])
S = pd.DataFrame(Cclip @ Cclip.T, index=CLASSES, columns=CLASSES).round(3)
S

> **Check.** With CLIP, the two closest classes are plant_pot and vase (cosine 0.864). A plant pot and a vase are both pots with plants or flowers in them.

## 5. Your turn: classify by the nearest photo

> **Your turn.** For each test photo, take the label of the **one** training photo with the highest similarity (ResNet-18 embeddings `Xtr`, `Xte`). Put the share of test photos where this label is right in `nn_accuracy` (a number from 0 to 1).

In [ ]:
nearest = (Xte @ Xtr.T).argmax(1)
nn_accuracy = float(np.mean(train["label"].values[nearest] == test["label"].values))
print(round(nn_accuracy, 3))

In [ ]:
expect('nn_accuracy (rounded)', None if nn_accuracy is ... else round(nn_accuracy, 3), 0.831, tolerance=0.02)

## 6. Failure case: a label error looks like a neighbour

Some training labels are wrong on purpose (a problem we planted). When a photo's nearest neighbours all have another label, look at it: it may be a label error, not a model error. Lesson 4 of this module finds them with confident learning.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [CLIP zero-shot](https://learning.nextia-ai.com/courses/vision/m03/clip-zero-shot/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.